# XWOD starter: train YOLO and make a submission

This notebook walks through the full competition workflow on **XWOD (eXtreme Weather Object Detection)**:

1. Load the data and look at labeled examples
2. Train a YOLO detector with Ultralytics
3. Check it on the validation split
4. **Create `submission.csv` in the competition format**
5. *(Optional)* Score yourself on the validation split with the **exact leaderboard metric**
6. Submit

The demo trains a small YOLO11n for 5 epochs, which takes a few minutes on a Kaggle GPU. Change the settings in section 4 to get a stronger model.

**Before you run:** in the notebook's settings panel, turn on **Accelerator → GPU** and **Internet → On**. Internet is needed to install Ultralytics and download pretrained weights.

> Paper: [XWOD: A Real-World Benchmark for Object Detection under Extreme Weather Conditions](https://arxiv.org/abs/2605.11521)

## 1. Setup

In [ ]:
%pip install -q -U ultralytics

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import Image as IPyImage, display
from ultralytics import YOLO


def find_data_root():
    # The competition folder is the one under /kaggle/input that contains data.yaml.
    for pattern in ("*/data.yaml", "*/*/data.yaml"):
        hits = sorted(Path("/kaggle/input").glob(pattern))
        if hits:
            return hits[0].parent
    # Outside Kaggle: point XWOD_DATA at the folder with data.yaml.
    return Path(os.environ.get("XWOD_DATA", "../input/xwod")).resolve()


DATA_ROOT = find_data_root()
DATA_YAML = DATA_ROOT / "data.yaml"
TEST_DIR = DATA_ROOT / "test" / "images"
VALID_DIR = DATA_ROOT / "valid" / "images"
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
SAMPLE_SUB = next((p for p in (DATA_ROOT / "sample_submission.csv", DATA_ROOT.parent / "sample_submission.csv")
                   if p.exists()), None)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert DATA_YAML.exists(), f"data.yaml not found under {DATA_ROOT}"
print("data      :", DATA_ROOT)
print("sample sub:", SAMPLE_SUB)
print("output    :", WORK)
print("device    :", DEVICE)

## 2. Dataset overview

There are six classes. The weather condition is the filename prefix, for example `snow_train_00012.jpg`. The test split has **images only**: its labels are hidden and used for the leaderboard.

In [ ]:
cfg = yaml.safe_load(DATA_YAML.read_text())
CLASSES = cfg["names"]
print("Classes:", dict(enumerate(CLASSES)))

WEATHERS = ["heavy_rain", "snow", "fog", "dust", "flooding", "tornado", "wildfire"]

def weather_of(name):
    return next((w for w in WEATHERS if name.startswith(w + "_")), "unknown")

rows = []
for split in ["train", "valid", "test"]:
    for f in (DATA_ROOT / split / "images").iterdir():
        rows.append((split, weather_of(f.name)))
counts = pd.DataFrame(rows, columns=["split", "weather"]).value_counts().unstack(0).fillna(0).astype(int)
counts.loc["total"] = counts.sum()
counts[["train", "valid", "test"]]

## 3. Look at labeled examples

Each image has a label file with one line per object: `class x_center y_center width height`, normalized to [0, 1].

In [ ]:
def read_labels(label_path):
    # Returns a list of (class, cx, cy, w, h). Skips empty lines (some files use \r\n or are empty).
    if not label_path.exists():
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            boxes.append((int(float(parts[0])), *map(float, parts[1:])))
    return boxes


def draw_boxes(img, boxes, color=(0, 255, 0)):
    H, W = img.shape[:2]
    for c, cx, cy, w, h in boxes:
        x0, y0 = int((cx - w / 2) * W), int((cy - h / 2) * H)
        x1, y1 = int((cx + w / 2) * W), int((cy + h / 2) * H)
        cv2.rectangle(img, (x0, y0), (x1, y1), color, max(2, W // 400))
        cv2.putText(img, CLASSES[c], (x0, max(14, y0 - 4)), cv2.FONT_HERSHEY_SIMPLEX,
                    max(0.6, W / 1600), color, 2)
    return img


def show_row(images, titles, height=300):
    panels = []
    for img, title in zip(images, titles):
        h, w = img.shape[:2]
        panel = cv2.resize(img, (int(w * height / h), height))
        cv2.putText(panel, title, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        panels.append(panel)
    ok, buf = cv2.imencode(".jpg", cv2.hconcat(panels))
    display(IPyImage(data=buf.tobytes()))


for row in (["heavy_rain", "snow", "fog", "dust"], ["flooding", "tornado", "wildfire"]):
    paths = [next((DATA_ROOT / "train" / "images").glob(f"{w}_*")) for w in row]
    imgs = [draw_boxes(cv2.imread(str(p)), read_labels(p.parent.parent / "labels" / f"{p.stem}.txt")) for p in paths]
    show_row(imgs, row)

## 4. Train

`yolo11n.pt` with 5 epochs is a quick demo. For a stronger model:

- **Bigger model:** try `yolo11s.pt`, `yolo11m.pt`, `yolov8m.pt` or `yolo26m.pt`.
- **More epochs:** the paper uses 100.
- **Larger `IMGSZ`:** 960 or 1280 helps with small, distant objects. Many images are 1080p or 4K.

You may see a warning that the label cache can't be saved: `/kaggle/input` is read-only. It's harmless.

In [ ]:
MODEL = "yolo11n.pt"   # try yolo11s.pt / yolo11m.pt / yolov8m.pt / yolo26m.pt
EPOCHS = 5             # demo; the paper uses 100
IMGSZ = 640

model = YOLO(MODEL)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(WORK / "runs"),
    name=Path(MODEL).stem,
    exist_ok=True,
    plots=False,
)
BEST = Path(train_results.save_dir) / "weights" / "best.pt"
print("best weights:", BEST)

## 5. Validate

Ultralytics reports mAP50 and mAP50-95 on the `valid` split. These are a quick sanity check; section 8 scores the validation split with the exact leaderboard metric.

In [ ]:
best_model = YOLO(BEST)
metrics = best_model.val(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE, plots=False, verbose=False)
print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")

## 6. Look at predictions on test images

In [ ]:
paths = [next(TEST_DIR.glob(f"{w}_*")) for w in ["heavy_rain", "snow", "fog", "wildfire"]]
preds = best_model.predict([str(p) for p in paths], imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
show_row([r.plot() for r in preds], [p.name for p in paths])

## 7. Create the submission file

The submission needs **one row per test image**. All boxes for the image go in one `PredictionString`, as groups of six values:

```
class confidence x_center y_center width height
```

- Coordinates are **normalized to [0, 1]**, the same format as the training labels. Ultralytics gives these directly as `r.boxes.xywhn`.
- An image with **no detections** gets the word `none`. Kaggle rejects empty cells.
- `image_id` is the filename without its extension.

```
image_id,PredictionString
dust_test_00001,1 0.9300 0.507135 0.808787 0.131454 0.153740 0 0.4012 0.047800 0.789100 0.026700 0.208200
dust_test_00002,none
```

Two settings matter for mAP:

- **`conf=0.001`:** keep low-confidence boxes. mAP rewards high recall, and low-confidence boxes rarely hurt.
- **`max_det=100`:** only the 100 most confident boxes per image are scored.

**Watch out:** when you pass a *list* of files to `predict()`, Ultralytics names the results `image0.jpg`, `image1.jpg`, …. Results come back in the same order as the inputs, so pair them with the input filenames rather than reading `r.path`.

In [ ]:
CONF = 0.001
MAX_DET = 100
BATCH = 16


def boxes_to_string(boxes):
    # Turn one image's Ultralytics boxes into a PredictionString.
    if boxes is None or len(boxes) == 0:
        return "none"
    return " ".join(
        f"{int(c)} {p:.4f} {x:.6f} {y:.6f} {w:.6f} {h:.6f}"
        for c, p, (x, y, w, h) in zip(boxes.cls.tolist(), boxes.conf.tolist(), boxes.xywhn.tolist())
    )


def predict_folder(model, image_dir):
    # Predict every image in a folder; returns a DataFrame in submission format.
    files = sorted(p for p in Path(image_dir).iterdir() if p.suffix.lower() in IMG_EXTS)
    rows = []
    for i in range(0, len(files), BATCH):
        batch = [str(f) for f in files[i:i + BATCH]]
        results = model.predict(batch, imgsz=IMGSZ, conf=CONF, max_det=MAX_DET, device=DEVICE, verbose=False)
        for f, r in zip(batch, results):  # same order as the inputs
            rows.append((Path(f).stem, boxes_to_string(r.boxes)))
        done = min(i + BATCH, len(files))
        if (i // BATCH) % 25 == 0 or done == len(files):
            print(f"{done}/{len(files)} images")
    return pd.DataFrame(rows, columns=["image_id", "PredictionString"])


submission = predict_folder(best_model, TEST_DIR)
submission.head()

Check the file before submitting. These checks catch the most common errors: missing images, duplicate rows and empty cells.

In [ ]:
assert submission["image_id"].is_unique, "duplicate image_id rows"
assert submission["PredictionString"].notna().all() and (submission["PredictionString"] != "").all(), "empty cells"

if SAMPLE_SUB is not None:
    sample = pd.read_csv(SAMPLE_SUB)
    missing = set(sample["image_id"]) - set(submission["image_id"])
    extra = set(submission["image_id"]) - set(sample["image_id"])
    assert not missing, f"{len(missing)} test images missing, e.g. {sorted(missing)[:3]}"
    assert not extra, f"{len(extra)} unknown image_ids, e.g. {sorted(extra)[:3]}"
    submission = submission.set_index("image_id").loc[sample["image_id"]].reset_index()  # same order as sample

n_boxes = submission["PredictionString"].map(lambda s: 0 if s == "none" else len(s.split()) // 6)
print(f"{len(submission)} images, {n_boxes.sum()} boxes, {(n_boxes == 0).sum()} images with 'none'")

submission.to_csv(WORK / "submission.csv", index=False)
print("saved", WORK / "submission.csv")

## 8. (Optional) Score yourself with the official metric

The test labels are hidden, but the `valid` split has labels. The next cell saves the **exact metric code used by the leaderboard** as `xwod_map.py`. The following cells then score your model on the validation split. The output includes per-class and per-weather results, which the leaderboard doesn't show.

A perfect submission scores 0.995, not 1.0. That's how Ultralytics computes AP, and the leaderboard uses the same method.

In [ ]:
%%writefile xwod_map.py
"""XWOD object-detection metric (COCO-style mAP) for Kaggle.

Kaggle's built-in metrics need a fixed number of values per row. Detection
output does not have that, so each test image is one row and all of its boxes
are packed into a single space-separated ``PredictionString``:

    submission.csv
        image_id,PredictionString
        dust_test_00001,1 0.91 0.5078 0.8025 0.1333 0.1530 0 0.40 0.0478 0.7891 0.0267 0.2082
        wildfire_test_00117,none

    Each box is 6 tokens: ``class conf cx cy w h``, with box coordinates
    normalized to [0, 1] (YOLO format, same as the dataset labels).
    Use the word ``none`` for an image with no detections. Kaggle rejects
    empty cells ("Submission contains null values"); an empty value is still
    accepted by this code for local use.

    solution.csv (host only)
        image_id,PredictionString,weather,Usage
        dust_test_00001,1 0.5078 0.8025 0.1333 0.1530 ...,dust,Public

    Ground-truth boxes are 5 tokens: ``class cx cy w h``.

The score is mAP50-95: AP averaged over IoU thresholds 0.50:0.05:0.95 and over
the classes present in the ground truth. The matching and AP code is a numpy
port of Ultralytics 8.4 (``DetectionValidator.match_predictions``,
``metrics.ap_per_class``, ``metrics.compute_ap``), so leaderboard numbers are
comparable to ``model.val()`` results reported in the XWOD paper. As in
Ultralytics, AP integrates 101 interpolated points with the trapezoid rule, so
a perfect submission scores 0.995 (not 1.0).

Normalized coordinates give the same IoU as pixel coordinates (IoU does not
change when x and y are scaled independently), so image sizes are not needed.

Only numpy and pandas are used.
"""

import numpy as np
import pandas as pd

CLASS_NAMES = ["person", "car", "truck", "motorcycle", "bus", "bike"]
NUM_CLASSES = len(CLASS_NAMES)
PRED_COL = "PredictionString"
WEATHER_COL = "weather"
USAGE_COL = "Usage"
NO_BOX = "none"                      # write this for an image with no detections
NO_BOX_TOKENS = {"", NO_BOX}


class ParticipantVisibleError(Exception):
    pass


# ---------------------------------------------------------------------------
# Parsing
# ---------------------------------------------------------------------------

def _is_empty(value) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and np.isnan(value):
        return True
    return str(value).strip().lower() in NO_BOX_TOKENS


def _parse_boxes(value, n_fields: int, image_id, error_cls, what: str) -> np.ndarray:
    """Parse one PredictionString into an (N, n_fields) float64 array."""
    if _is_empty(value):
        return np.zeros((0, n_fields), dtype=np.float64)
    tokens = str(value).split()
    if len(tokens) % n_fields != 0:
        raise error_cls(
            f"{what} for image_id '{image_id}' has {len(tokens)} values, which is not a "
            f"multiple of {n_fields}. Expected repeated groups of "
            + ("'class conf cx cy w h'." if n_fields == 6 else "'class cx cy w h'.")
        )
    try:
        arr = np.asarray(tokens, dtype=np.float64).reshape(-1, n_fields)
    except ValueError:
        raise error_cls(f"{what} for image_id '{image_id}' contains a value that is not a number.")
    if not np.isfinite(arr).all():
        raise error_cls(f"{what} for image_id '{image_id}' contains NaN or infinite values.")
    cls = arr[:, 0]
    if (cls != np.round(cls)).any() or (cls < 0).any() or (cls >= NUM_CLASSES).any():
        raise error_cls(
            f"{what} for image_id '{image_id}' has a class id outside 0..{NUM_CLASSES - 1}."
        )
    wh = arr[:, -2:]
    if (wh < 0).any():
        raise error_cls(f"{what} for image_id '{image_id}' has a box with negative width or height.")
    return arr


def _cxcywh_to_xyxy(b: np.ndarray) -> np.ndarray:
    out = np.empty_like(b)
    out[:, 0] = b[:, 0] - b[:, 2] / 2
    out[:, 1] = b[:, 1] - b[:, 3] / 2
    out[:, 2] = b[:, 0] + b[:, 2] / 2
    out[:, 3] = b[:, 1] + b[:, 3] / 2
    return out


# ---------------------------------------------------------------------------
# Ultralytics-equivalent core (numpy)
# ---------------------------------------------------------------------------

# Ultralytics adds eps=1e-7 to the union in *pixel* units. Boxes here are normalized
# (areas are ~1e-4 for small objects), so a much smaller eps keeps IoU equivalent.
IOU_EPS = 1e-12


def _box_iou_xyxy(box1: np.ndarray, box2: np.ndarray, eps: float = IOU_EPS) -> np.ndarray:
    """IoU matrix (len(box1), len(box2)); same formula as ultralytics box_iou."""
    a1, a2 = box1[:, None, :2], box1[:, None, 2:]
    b1, b2 = box2[None, :, :2], box2[None, :, 2:]
    inter = np.clip(np.minimum(a2, b2) - np.maximum(a1, b1), 0, None).prod(2)
    area1 = (box1[:, 2:] - box1[:, :2]).prod(1)[:, None]
    area2 = (box2[:, 2:] - box2[:, :2]).prod(1)[None, :]
    return inter / (area1 + area2 - inter + eps)


def _match_predictions(pred_classes, true_classes, iou, iouv) -> np.ndarray:
    """Greedy matching, identical to DetectionValidator.match_predictions (non-scipy).

    iou has shape (n_labels, n_preds). Returns bool array (n_preds, len(iouv)).
    """
    correct = np.zeros((pred_classes.shape[0], len(iouv)), dtype=bool)
    iou = iou * (true_classes[:, None] == pred_classes[None, :])
    for i, threshold in enumerate(iouv):
        matches = np.array(np.nonzero(iou >= threshold)).T
        if matches.shape[0]:
            if matches.shape[0] > 1:
                matches = matches[iou[matches[:, 0], matches[:, 1]].argsort()[::-1]]
                matches = matches[np.unique(matches[:, 1], return_index=True)[1]]
                matches = matches[np.unique(matches[:, 0], return_index=True)[1]]
            correct[matches[:, 1].astype(int), i] = True
    return correct


_trapezoid = getattr(np, "trapezoid", None) or getattr(np, "trapz")


def _compute_ap(recall: np.ndarray, precision: np.ndarray) -> float:
    """101-point interpolated AP, identical to ultralytics metrics.compute_ap."""
    mrec = np.concatenate(([0.0], recall, [recall[-1] if len(recall) else 1.0], [1.0]))
    mpre = np.concatenate(([1.0], precision, [0.0], [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_trapezoid(np.interp(x, mrec, mpre), x))


def _ap_per_class(tp, conf, pred_cls, target_cls, eps: float = 1e-16):
    """AP matrix (n_gt_classes, n_iou) and its classes; same as ultralytics ap_per_class."""
    i = np.argsort(-conf)
    tp, conf, pred_cls = tp[i], conf[i], pred_cls[i]
    unique_classes, nt = np.unique(target_cls, return_counts=True)
    ap = np.zeros((unique_classes.shape[0], tp.shape[1]))
    for ci, c in enumerate(unique_classes):
        i = pred_cls == c
        n_l = nt[ci]
        n_p = i.sum()
        if n_p == 0 or n_l == 0:
            continue
        fpc = (1 - tp[i]).cumsum(0)
        tpc = tp[i].cumsum(0)
        recall = tpc / (n_l + eps)
        precision = tpc / (tpc + fpc)
        for j in range(tp.shape[1]):
            ap[ci, j] = _compute_ap(recall[:, j], precision[:, j])
    return ap, unique_classes


# ---------------------------------------------------------------------------
# Scoring
# ---------------------------------------------------------------------------

def _iou_thresholds(iou_start, iou_stop, iou_step) -> np.ndarray:
    n = int(round((iou_stop - iou_start) / iou_step)) + 1
    return np.linspace(iou_start, iou_stop, n)


def _collect_stats(gt_strings, pred_strings, image_ids, iouv, max_dets):
    """Per-image matching; returns concatenated (tp, conf, pred_cls, target_cls)."""
    tps, confs, pcls, tcls = [], [], [], []
    for image_id, gt_s, pred_s in zip(image_ids, gt_strings, pred_strings):
        gt = _parse_boxes(gt_s, 5, image_id, ValueError, "Solution")
        pred = _parse_boxes(pred_s, 6, image_id, ParticipantVisibleError, PRED_COL)
        if len(pred) > max_dets:
            pred = pred[np.argsort(-pred[:, 1], kind="stable")[:max_dets]]

        gt_cls = gt[:, 0]
        tcls.append(gt_cls)
        if len(pred) == 0:
            continue
        pred_box = np.clip(_cxcywh_to_xyxy(pred[:, 2:6]), 0.0, 1.0)
        correct = np.zeros((len(pred), len(iouv)), dtype=bool)
        if len(gt):
            iou = _box_iou_xyxy(_cxcywh_to_xyxy(gt[:, 1:5]), pred_box)
            correct = _match_predictions(pred[:, 0], gt_cls, iou, iouv)
        tps.append(correct)
        confs.append(pred[:, 1])
        pcls.append(pred[:, 0])

    n_iou = len(iouv)
    tp = np.concatenate(tps) if tps else np.zeros((0, n_iou), dtype=bool)
    conf = np.concatenate(confs) if confs else np.zeros(0)
    pred_cls = np.concatenate(pcls) if pcls else np.zeros(0)
    target_cls = np.concatenate(tcls) if tcls else np.zeros(0)
    return tp, conf, pred_cls, target_cls


def _align(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> pd.DataFrame:
    """Return the solution with the participant's PredictionString attached as '_pred'."""
    if row_id_column_name not in submission.columns:
        raise ParticipantVisibleError(f"Submission is missing the '{row_id_column_name}' column.")
    if PRED_COL not in submission.columns:
        raise ParticipantVisibleError(f"Submission is missing the '{PRED_COL}' column.")
    sub = submission[[row_id_column_name, PRED_COL]].copy()
    sub[row_id_column_name] = sub[row_id_column_name].astype(str)
    dup = sub[row_id_column_name].duplicated()
    if dup.any():
        example = sub.loc[dup, row_id_column_name].iloc[0]
        raise ParticipantVisibleError(
            f"Submission has {int(dup.sum())} duplicated {row_id_column_name} values (e.g. '{example}'). "
            "Use one row per image with all of its boxes in a single PredictionString."
        )
    sol = solution.copy()
    sol[row_id_column_name] = sol[row_id_column_name].astype(str)
    missing = ~sol[row_id_column_name].isin(sub[row_id_column_name])
    if missing.any():
        example = sol.loc[missing, row_id_column_name].iloc[0]
        raise ParticipantVisibleError(
            f"Submission is missing {int(missing.sum())} test images (e.g. '{example}'). "
            "Include every image_id from sample_submission.csv, with PredictionString 'none' "
            "if there are no detections."
        )
    # Extra ids are ignored: Kaggle may pass the full submission with a Public/Private subset.
    sub = sub.rename(columns={PRED_COL: "_pred"})
    return sol.merge(sub, on=row_id_column_name, how="left")


def _map_from_frame(df, row_id_column_name, iouv, max_dets):
    tp, conf, pred_cls, target_cls = _collect_stats(
        df[PRED_COL].tolist(), df["_pred"].tolist(), df[row_id_column_name].tolist(), iouv, max_dets
    )
    if target_cls.size == 0:
        return np.zeros((0, len(iouv))), np.zeros(0)
    return _ap_per_class(tp, conf, pred_cls, target_cls)


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    iou_start: float = 0.5,
    iou_stop: float = 0.95,
    iou_step: float = 0.05,
    max_dets: int = 100,
    primary: str = "map50_95",
) -> float:
    """XWOD leaderboard score: mAP50-95 (or mAP50 with primary='map50').

    Kaggle calls this separately for the Public and Private rows of the solution.

    >>> sol = pd.DataFrame({
    ...     "image_id": ["a", "b"],
    ...     "PredictionString": ["1 0.5 0.5 0.2 0.2", "0 0.3 0.3 0.1 0.2"],
    ... })
    >>> perfect = pd.DataFrame({
    ...     "image_id": ["a", "b"],
    ...     "PredictionString": ["1 0.9 0.5 0.5 0.2 0.2", "0 0.8 0.3 0.3 0.1 0.2"],
    ... })
    >>> round(score(sol, perfect, "image_id"), 4)   # Ultralytics-style AP tops out at 0.995
    0.995
    >>> empty = pd.DataFrame({"image_id": ["a", "b"], "PredictionString": ["none", "none"]})
    >>> score(sol, empty, "image_id")
    0.0
    """
    if primary not in ("map50_95", "map50"):
        raise ValueError(f"Unknown primary metric '{primary}'.")
    iouv = _iou_thresholds(iou_start, iou_stop, iou_step)
    df = _align(solution, submission, row_id_column_name)
    ap, _ = _map_from_frame(df, row_id_column_name, iouv, int(max_dets))
    if ap.size == 0:
        return 0.0
    return float(ap[:, 0].mean() if primary == "map50" else ap.mean())


def evaluate(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str = "image_id",
    max_dets: int = 100,
) -> dict:
    """Detailed local report: overall, per-class, per-weather and per-Usage mAP."""
    iouv = _iou_thresholds(0.5, 0.95, 0.05)
    df = _align(solution, submission, row_id_column_name)

    def summary(frame):
        ap, classes = _map_from_frame(frame, row_id_column_name, iouv, max_dets)
        if ap.size == 0:
            return {"mAP50": 0.0, "mAP50-95": 0.0, "images": len(frame)}, {}
        per_class = {
            CLASS_NAMES[int(c)]: {"AP50": float(ap[k, 0]), "AP50-95": float(ap[k].mean())}
            for k, c in enumerate(classes)
        }
        return {"mAP50": float(ap[:, 0].mean()), "mAP50-95": float(ap.mean()), "images": len(frame)}, per_class

    overall, per_class = summary(df)
    report = {"overall": overall, "per_class": per_class}
    for col, key in ((WEATHER_COL, "per_weather"), (USAGE_COL, "per_usage")):
        if col in df.columns:
            report[key] = {str(v): summary(g)[0] for v, g in df.groupby(col, sort=True)}
    return report

In [ ]:
import importlib
import xwod_map
importlib.reload(xwod_map)


def labels_to_solution(image_dir):
    # Build a solution table (the leaderboard's answer-key format) from YOLO label files.
    rows = []
    for img in sorted(p for p in Path(image_dir).iterdir() if p.suffix.lower() in IMG_EXTS):
        lines = (img.parent.parent / "labels" / f"{img.stem}.txt")
        boxes = []
        for line in (lines.read_text().splitlines() if lines.exists() else []):
            v = [float(x) for x in line.split()]
            if len(v) == 5:
                c, cx, cy, w, h = v
            elif len(v) > 5:  # polygon -> its bounding box
                xs, ys = v[1::2], v[2::2]
                c, cx, cy, w, h = v[0], (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
            else:
                continue
            # clip to the image, like the competition answer key
            x0, y0, x1, y1 = (max(0, cx - w / 2), max(0, cy - h / 2), min(1, cx + w / 2), min(1, cy + h / 2))
            boxes.append(f"{int(c)} {(x0 + x1) / 2:.10g} {(y0 + y1) / 2:.10g} {x1 - x0:.10g} {y1 - y0:.10g}")
        rows.append((img.stem, " ".join(boxes) or "none", weather_of(img.name)))
    return pd.DataFrame(rows, columns=["image_id", "PredictionString", "weather"])


valid_solution = labels_to_solution(VALID_DIR)
valid_pred = predict_folder(best_model, VALID_DIR)
report = xwod_map.evaluate(valid_solution, valid_pred)

print(f"Validation mAP50-95 (leaderboard metric): {report['overall']['mAP50-95']:.4f}")
print(f"Validation mAP50                        : {report['overall']['mAP50']:.4f}")

In [ ]:
per_class = pd.DataFrame(report["per_class"]).T.round(4)
per_weather = pd.DataFrame(report["per_weather"]).T[["images", "mAP50", "mAP50-95"]]
per_weather[["mAP50", "mAP50-95"]] = per_weather[["mAP50", "mAP50-95"]].astype(float).round(4)
per_weather["images"] = per_weather["images"].astype(int)
display(per_class)
display(per_weather.sort_values("mAP50-95"))

## 9. Submit

`submission.csv` is saved in `/kaggle/working`. To submit it:

- **From this notebook:** click **Save Version** and choose *Save & Run All*. When it finishes, open the saved version, go to the **Output** tab, and click **Submit**.
- **Or upload it:** download `submission.csv` from the Output panel and upload it on the competition's **Submit Prediction** page.

The public leaderboard scores about 30% of the test images. Your final rank uses the other 70%, so don't over-tune to the public score.

## 10. Ideas to improve

- **Bigger models and longer training:** YOLO11m or YOLO26m for 50–100 epochs.
- **Higher resolution:** `IMGSZ=960` or `1280` for small objects. Use the same value for training and prediction.
- **Use all labeled data:** once you've picked your settings, train on `train` + `valid` together.
- **Test-time augmentation:** `predict(..., augment=True)`.
- **Rare classes:** bus and bike have few examples. Check the per-class table above.
- **Hard conditions:** check which weather scores lowest in the per-weather table and look at those images.

**Rule reminder:** don't use XWOD test labels from any other source, such as the earlier public dataset release. Submissions must come from your model's predictions.